# House Prices Modeling

## 1. Imports

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_log_error

## 2. RMSLE Function

In [3]:
def compute_rmsle(y_test: np.ndarray, y_pred: np.ndarray, precision: int = 2) -> float:
    """Note: y_pred values must be positive (log is applied)."""
    y_pred = np.maximum(y_pred, 1)
    rmsle = np.sqrt(mean_squared_log_error(y_test, y_pred))
    return round(rmsle, precision)

## 3. Dataset Loading

In [4]:
train_path = "../data/train.csv"
test_path = "../data/test.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (1460, 81)
Test shape: (1459, 80)


In [5]:
train_df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


## 4. Data Exploration

In [6]:
train_df[["SalePrice", "GrLivArea", "TotalBsmtSF", "Neighborhood", "HouseStyle"]].head()

,SalePrice,GrLivArea,TotalBsmtSF,Neighborhood,HouseStyle
0,208500,1710,856,CollgCr,2Story
1,181500,1262,1262,Veenker,1Story
2,223500,1786,920,CollgCr,2Story
3,140000,1717,756,Crawfor,2Story
4,250000,2198,1145,NoRidge,2Story


In [7]:
train_df[["SalePrice", "GrLivArea", "TotalBsmtSF"]].describe()

,SalePrice,GrLivArea,TotalBsmtSF
count,1460.000000,1460.000000,1460.000000
mean,180921.195890,1515.463699,1057.429452
std,79442.502883,525.480383,438.705324
min,34900.000000,334.000000,0.000000
25%,129975.000000,1129.500000,795.750000
50%,163000.000000,1464.000000,991.500000
75%,214000.000000,1776.750000,1298.250000
max,755000.000000,5642.000000,6110.000000


In [8]:
train_df[["Neighborhood", "HouseStyle"]].describe()

,Neighborhood,HouseStyle
count,1460,1460
unique,25,8
top,NAmes,1Story
freq,225,726


## 5. Feature Selection

In [9]:
selected_features = ["GrLivArea", "TotalBsmtSF", "Neighborhood", "HouseStyle"]
continuous_features = ["GrLivArea", "TotalBsmtSF"]
categorical_features = ["Neighborhood", "HouseStyle"]
target_column = "SalePrice"

df = train_df[selected_features + [target_column]].copy()

print("Selected features:", selected_features)
df.head()

Selected features: ['GrLivArea', 'TotalBsmtSF', 'Neighborhood', 'HouseStyle']


,GrLivArea,TotalBsmtSF,Neighborhood,HouseStyle,SalePrice
0,1710,856,CollgCr,2Story,208500
1,1262,1262,Veenker,1Story,181500
2,1786,920,CollgCr,2Story,223500
3,1717,756,Crawfor,2Story,140000
4,2198,1145,NoRidge,2Story,250000


## 6. Missing Values Handling

In [10]:
df.isnull().sum()

GrLivArea       0
TotalBsmtSF     0
Neighborhood    0
HouseStyle      0
SalePrice       0
dtype: int64

In [11]:
for col in continuous_features:
    df[col] = df[col].fillna(df[col].median())

for col in categorical_features:
    df[col] = df[col].fillna(df[col].mode()[0])

df.isnull().sum()

GrLivArea       0
TotalBsmtSF     0
Neighborhood    0
HouseStyle      0
SalePrice       0
dtype: int64

## 7. Train / Validation Split

In [12]:
X = df[selected_features].copy()
y = df[target_column].copy()

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train shape:", X_train.shape)
print("X_valid shape:", X_valid.shape)
print("y_train shape:", y_train.shape)
print("y_valid shape:", y_valid.shape)

X_train shape: (1168, 4)
X_valid shape: (292, 4)
y_train shape: (1168,)
y_valid shape: (292,)


## 8. Continuous Features Scaling

In [13]:
X_train_processed = X_train.copy()
X_valid_processed = X_valid.copy()

scaling_parameters = {}

for col in continuous_features:
    mean_value = X_train_processed[col].mean()
    std_value = X_train_processed[col].std()

    scaling_parameters[col] = {
        "mean": mean_value,
        "std": std_value
    }

    X_train_processed[col] = (X_train_processed[col] - mean_value) / std_value
    X_valid_processed[col] = (X_valid_processed[col] - mean_value) / std_value

scaling_parameters

{'GrLivArea': {'mean': np.float64(1527.4015410958905),
  'std': np.float64(524.4326862400336)},
 'TotalBsmtSF': {'mean': np.float64(1061.771404109589),
  'std': np.float64(440.6763302880878)}}

In [14]:
X_train_processed.head()

,GrLivArea,TotalBsmtSF,Neighborhood,HouseStyle
254,-0.406919,0.572367,NAmes,1Story
1066,0.083135,-0.596291,Gilbert,2Story
638,-1.394653,-0.603099,Edwards,1Story
799,0.458779,-0.750599,SWISU,1.5Fin
380,0.311953,-0.081174,SWISU,1.5Fin


## 9. Categorical Features Encoding

In [15]:
X_train_processed = pd.get_dummies(
    X_train_processed,
    columns=categorical_features,
    drop_first=False
)

X_valid_processed = pd.get_dummies(
    X_valid_processed,
    columns=categorical_features,
    drop_first=False
)

X_valid_processed = X_valid_processed.reindex(
    columns=X_train_processed.columns,
    fill_value=0
)

print("Processed train shape:", X_train_processed.shape)
print("Processed valid shape:", X_valid_processed.shape)

Processed train shape: (1168, 35)
Processed valid shape: (292, 35)


In [16]:
X_train_processed.head()

,GrLivArea,TotalBsmtSF,Neighborhood_Blmngtn,Neighborhood_Blueste,Neighborhood_BrDale,Neighborhood_BrkSide,Neighborhood_ClearCr,Neighborhood_CollgCr,Neighborhood_Crawfor,Neighborhood_Edwards,...,Neighborhood_Timber,Neighborhood_Veenker,HouseStyle_1.5Fin,HouseStyle_1.5Unf,HouseStyle_1Story,HouseStyle_2.5Fin,HouseStyle_2.5Unf,HouseStyle_2Story,HouseStyle_SFoyer,HouseStyle_SLvl
254,-0.406919,0.572367,False,False,False,False,False,False,False,False,...,False,False,False,False,True,False,False,False,False,False
1066,0.083135,-0.596291,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
638,-1.394653,-0.603099,False,False,False,False,False,False,False,True,...,False,False,False,False,True,False,False,False,False,False
799,0.458779,-0.750599,False,False,False,False,False,False,False,False,...,False,False,True,False,False,False,False,False,False,False
380,0.311953,-0.081174,False,False,False,False,False,False,False,False,...,False,False,True,False,False,False,False,False,False,False


## 10. Model Training

In [17]:
model = LinearRegression()
model.fit(X_train_processed, y_train)

print("Model training completed.")

Model training completed.


## 11. Model Evaluation

In [18]:
y_pred = model.predict(X_valid_processed)
score = compute_rmsle(y_valid, y_pred)

print("Validation RMSLE:", score)

Validation RMSLE: 0.19


## 12. Conclusion

In this notebook, a house price prediction model was built using:
- 2 continuous features: `GrLivArea`, `TotalBsmtSF`
- 2 categorical features: `Neighborhood`, `HouseStyle`

The preprocessing was implemented manually:
- missing values were filled explicitly
- continuous variables were scaled manually
- categorical variables were encoded with one-hot encoding using pandas

A Linear Regression model was then trained and evaluated using RMSLE, which is the competition metric.